# TP4 - overfitting, data leakage and generalization

Working with the student performance dataset.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('hybrid_student_performance_1200.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
print('Shape:', df.shape)
print(df.head())


In [2]:
print('Missing values (%):')
print((df.isna().mean() * 100).sort_values(ascending=False).head(15))


## Part A - exploring the data


In [3]:
print('Dimensions:', df.shape)
print(df.describe(include='all').T)


In [4]:
print('Duplicates:', df.duplicated().sum())


Converting categorical columns to numbers.


In [5]:
cgpa_map = {'5.0 – 6.9': 5.95, '7.0 – 8.4': 7.70, '8.5 – 9.4': 8.95, '9.5 – 10.0': 9.75}
df['final_cgpa'] = df['cgpa_category'].map(cgpa_map)

study_map = {'Less than 1 hour': 0.5, '1–2 hours': 1.5, 'More than 2 hours': 2.5}
sleep_map = {'4–5 hours': 4.5, '6–7 hours': 6.5, 'More than 8 hours': 8.5}
attendance_map = {'Less than 50%': 40, '50% – 65%': 57.5, '50-65%': 57.5,
                  '66% – 75%': 70.5, '76% – 85%': 80.5, 'Above 85%': 92.5}
df['study_hours'] = df['study_hours_daily'].map(study_map)
df['sleep_hours'] = df['sleep_hours'].map(sleep_map)
df['attendance_rate'] = df['attendance_percentage'].map(attendance_map)

ordinal_3 = {'Rarely': 0, 'Sometimes': 1, 'Often': 2, 'Always': 3}
revision_map = {'Never': 0, 'Rarely': 1, 'Few times a week': 2, 'Daily': 3}
df['tasks_on_time'] = df['tasks_on_time'].map(ordinal_3)
df['assignments_on_time'] = df['assignments_on_time'].map(ordinal_3)
df['revision_frequency'] = df['revision_frequency'].map(revision_map)

first_time = df['timestamp'].min()
df['submission_day'] = (df['timestamp'] - first_time).dt.days
df = df.sort_values('timestamp').reset_index(drop=True)

print(df[['study_hours', 'sleep_hours', 'attendance_rate', 'final_cgpa', 'submission_day']].head())


In [6]:
numeric_features = ['study_hours', 'sleep_hours', 'attendance_rate', 'tasks_on_time', 'assignments_on_time', 'revision_frequency', 'daily_productivity', 'stress_level', 'age', 'submission_day']
corr = df[numeric_features + ['final_cgpa']].corr()['final_cgpa'].drop('final_cgpa').sort_values()
print(corr)


In [7]:
rolling = df.set_index('timestamp')['final_cgpa'].rolling('7D').mean()
plt.figure(figsize=(10, 4))
plt.plot(rolling.index, rolling.values)
plt.title('7-day rolling mean of final CGPA')
plt.xlabel('Submission date')
plt.ylabel('Final CGPA')
plt.grid(alpha=0.3)
plt.show()


## Part B - splitting and data leakage


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

X = df[numeric_features]
y = df['final_cgpa']

split = int(len(df) * 0.8)
chron_train = df.iloc[:split]
chron_test = df.iloc[split:]

X_train_chron = chron_train[numeric_features]
X_test_chron = chron_test[numeric_features]
y_train_chron = chron_train['final_cgpa']
y_test_chron = chron_test['final_cgpa']

print('Train size:', len(chron_train), 'Test size:', len(chron_test))


In [9]:
random_train, random_test, ry_train, ry_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree = Pipeline([('imp', SimpleImputer(strategy='median')), ('model', DecisionTreeRegressor(random_state=42))])
tree.fit(random_train, ry_train)
print('Random split - train RMSE:', np.sqrt(mean_squared_error(ry_train, tree.predict(random_train))))
print('Random split - test RMSE:', np.sqrt(mean_squared_error(ry_test, tree.predict(random_test))))


In [10]:
tree2 = Pipeline([('imp', SimpleImputer(strategy='median')), ('model', DecisionTreeRegressor(random_state=42))])
tree2.fit(X_train_chron, y_train_chron)
print('Chronological - train RMSE:', np.sqrt(mean_squared_error(y_train_chron, tree2.predict(X_train_chron))))
print('Chronological - test RMSE:', np.sqrt(mean_squared_error(y_test_chron, tree2.predict(X_test_chron))))


Trying to add duplicate rows and see if it leaks.


In [11]:
rng = np.random.default_rng(42)
augmented = df.copy()
n_dup = int(round(0.15 * len(df)))
chosen = rng.choice(df.index, size=n_dup, replace=False)
dups = df.loc[chosen].copy()
min_t = df['timestamp'].min()
max_t = df['timestamp'].max()
dups['timestamp'] = pd.to_datetime(rng.uniform(min_t.value, max_t.value, size=n_dup).tolist(), unit='ns')
augmented = pd.concat([augmented, dups], ignore_index=True).sort_values('timestamp')

split_a = int(len(augmented) * 0.8)
a_train = augmented.iloc[:split_a]
a_test = augmented.iloc[split_a:]

m = Pipeline([('imp', SimpleImputer(strategy='median')), ('tree', DecisionTreeRegressor(random_state=42))])
m.fit(a_train[numeric_features], a_train['final_cgpa'])
print('With duplicate leakage - test RMSE:', np.sqrt(mean_squared_error(a_test['final_cgpa'], m.predict(a_test[numeric_features]))))


In [12]:
cols_except_ts = [c for c in augmented.columns if c != 'timestamp']
aug_clean = augmented.drop_duplicates(subset=cols_except_ts)
aug_clean = aug_clean.sort_values('submission_day')
split_r = int(len(aug_clean) * 0.8)
r_train = aug_clean.iloc[:split_r]
r_test = aug_clean.iloc[split_r:]

m.fit(r_train[numeric_features], r_train['final_cgpa'])
print('Without duplicate leakage - test RMSE:', np.sqrt(mean_squared_error(r_test['final_cgpa'], m.predict(r_test[numeric_features]))))


Checking if fitting the scaler on the whole dataset leaks.


In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

imp = SimpleImputer(strategy='median')
full_imp = imp.fit_transform(df[numeric_features])
sc = StandardScaler()
full_sc = sc.fit_transform(full_imp)

knn_bad = KNeighborsRegressor(n_neighbors=10)
knn_bad.fit(full_sc[:split], y[:split])
print('Scaler fit on all data - test RMSE:', np.sqrt(mean_squared_error(y[split:], knn_bad.predict(full_sc[split:]))))

knn_good = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('knn', KNeighborsRegressor(n_neighbors=10))])
knn_good.fit(X_train_chron, y_train_chron)
print('Scaler inside pipeline - test RMSE:', np.sqrt(mean_squared_error(y_test_chron, knn_good.predict(X_test_chron))))


## Part C - overfitting


In [14]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

depth_rows = []
for depth in [1, 2, 3, 5, 10, 20, None]:
    model = Pipeline([('imp', SimpleImputer(strategy='median')), ('tree', DecisionTreeRegressor(max_depth=depth, random_state=42))])
    model.fit(X_train_chron, y_train_chron)
    depth_rows.append([str(depth), rmse(y_train_chron, model.predict(X_train_chron)), rmse(y_test_chron, model.predict(X_test_chron))])

depth_df = pd.DataFrame(depth_rows, columns=['Depth', 'Train RMSE', 'Test RMSE'])
print(depth_df.round(4))


In [15]:
plt.figure(figsize=(8, 4))
plt.plot(depth_df['Depth'].astype(str), depth_df['Train RMSE'], marker='o', label='Train RMSE')
plt.plot(depth_df['Depth'].astype(str), depth_df['Test RMSE'], marker='o', label='Test RMSE')
plt.xlabel('Tree depth')
plt.ylabel('RMSE')
plt.title('Overfitting vs tree depth')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Part D - improving generalization


In [16]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score


# effort index
effort_cols = ['study_hours', 'attendance_rate', 'tasks_on_time', 'assignments_on_time', 'revision_frequency']
divs = [2.5, 100, 2, 3, 3]
effort_index = pd.concat([df[c].div(d) for c, d in zip(effort_cols, divs)], axis=1).mean(axis=1)
df['effort_index'] = effort_index
chron_train['effort_index'] = effort_index.loc[chron_train.index]

et = chron_train[['effort_index']]
ttscv = TimeSeriesSplit(n_splits=5)

poly_rows = []
for degree in [1, 2, 3, 5, 9, 15]:
    model = Pipeline([('imp', SimpleImputer(strategy='median')), ('poly', PolynomialFeatures(degree=degree, include_bias=False)), ('sc', StandardScaler()), ('lr', LinearRegression())])
    scores = cross_val_score(model, et, y_train_chron, cv=ttscv, scoring='neg_mean_squared_error')
    poly_rows.append([degree, np.sqrt(-scores).mean()])

poly_df = pd.DataFrame(poly_rows, columns=['Degree', 'CV RMSE'])
print(poly_df.round(4))


In [17]:
ridge_rows = []
for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
    ridge = Pipeline([('poly', PolynomialFeatures(degree=5, include_bias=False)), ('sc', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    scores = cross_val_score(ridge, et, y_train_chron, cv=ttscv, scoring='neg_mean_squared_error')
    ridge_rows.append([alpha, np.sqrt(-scores).mean()])

ridge_df = pd.DataFrame(ridge_rows, columns=['Alpha', 'Ridge CV RMSE'])
print(ridge_df.round(4))


In [18]:
best_alpha = float(ridge_df.loc[ridge_df['Ridge CV RMSE'].idxmin(), 'Alpha'])
print('Best alpha:', best_alpha)


## Final answers


- Overfitting happens when the model learns the training data too well (even the noise) and does poorly on new data.
- Data leakage means information from the test set gets into training (e.g. fitting a scaler on the whole dataset), which makes results look better than they really are.
- Regularization like Ridge/Lasso penalizes large coefficients and helps generalization.
